In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import lightgbm as lgb

In [2]:
SEED = 42
np.random.seed(SEED)

In [3]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])

In [4]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    # ---- Bees ----
    data['total_bees'] = (
        data['honeybee'] +
        data['bumbles'] +
        data['andrena'] +
        data['osmia']
    )

    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)

    data['osmia_honeybee']   = data['osmia'] * data['honeybee']
    data['bumble_honeybee'] = data['bumbles'] * data['honeybee']
    data['andrena_osmia']   = data['andrena'] * data['osmia']

    # ---- Temperature ----
    data['temp_range'] = data['MaxOfUpperTRange'] - data['MinOfLowerTRange']

    data['avg_temp'] = (
        data['AverageOfUpperTRange'] +
        data['AverageOfLowerTRange']
    ) / 2

    data['temp_x_rain']  = data['avg_temp'] * data['RainingDays']
    data['temp_rain_ratio'] = data['avg_temp'] / (data['RainingDays'] + 1e-6)

    # ---- Statistics ----
    bee_cols = ['honeybee','bumbles','andrena','osmia']
    data['bees_mean'] = data[bee_cols].mean(axis=1)
    data['bees_std']  = data[bee_cols].std(axis=1)

    # ---- Non-linear ----
    for col in ['clonesize', 'total_bees', 'fruitmass', 'seeds']:
        data[f'log_{col}'] = np.log1p(data[col])
        data[f'{col}_sq'] = data[col] ** 2

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    # ---- Clustering ----
    cluster_cols = [
        'clonesize',
        'total_bees',
        'avg_temp',
        'RainingDays',
        'fruitmass'
    ]

    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])

        kmeans = KMeans(
            n_clusters=8,      # ⬅️ مهم
            random_state=SEED,
            n_init=30
        )
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)

    data['cluster_bees'] = data['cluster'] * data['total_bees']

    return data, kmeans, scaler


train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)


In [5]:
X = train_fe.drop(columns=['yield'])
y = np.log1p(train_fe['yield'])   #

In [6]:
params = {
    'objective': 'regression_l1',
    'metric': 'mae',

    'learning_rate': 0.02,
    'num_leaves': 128,
    'min_data_in_leaf': 20,

    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 1,

    'lambda_l1': 0.5,
    'lambda_l2': 0.5,
    'min_gain_to_split': 0.01,

    'boosting': 'gbdt',

    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,

    'verbosity': -1,
    'seed': SEED
}

# ===============================
# CV
# ===============================
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

oof = np.zeros(len(X))
test_preds = np.zeros(len(test_fe))
maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'Fold {fold}/10')

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    train_set = lgb.Dataset(X_tr, y_tr)
    val_set   = lgb.Dataset(X_val, y_val)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=7000,
        valid_sets=[val_set],
        callbacks=[
            lgb.early_stopping(300, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    val_pred = np.expm1(model.predict(X_val))
    oof[val_idx] = val_pred

    fold_mae = mean_absolute_error(
        np.expm1(y_val),
        val_pred
    )
    maes.append(fold_mae)

    test_preds += np.expm1(model.predict(test_fe)) / kf.n_splits

print('\n🔥 OOF MAE:', np.mean(maes), '±', np.std(maes))


Fold 1/10


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

🔥 OOF MAE: 245.96458534667477 ± 6.394064752040805


In [7]:
final_test_preds = np.clip(
    test_preds,
    train['yield'].min(),
    train['yield'].max()
)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': final_test_preds
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,yield
0,15000,7524.503957
1,15001,5887.968910
2,15002,6474.511228
3,15003,4695.878875
4,15004,5913.570612
